In [ ]:
import pandas as pd

df = pd.read_csv("../House_Rent_Dataset.csv")


df["Floor"] = df["Floor"].astype(str).str.split(" out of").str[0]
df["Floor"] = df["Floor"].replace({"Ground": 0, "Upper Basement": -1, "Lower Basement": -2})
df["Floor"] = pd.to_numeric(df["Floor"], errors="coerce").fillna(0)

print(df.shape)
df.head()

(4746, 12)


,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,0,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [2]:
def filter_listings(city, max_rent):
    return df[(df["City"] == city) & (df["Rent"] <= max_rent)]

print(len(filter_listings("Mumbai", 50000)))
print(len(filter_listings("Mumbai", 20000)))
print(len(filter_listings("Kolkata", 8000)))
print(len(filter_listings("Kolkata", 20000)))

478
111
256
469


In [3]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df["log_rent"] = np.log1p(df["Rent"])

scaler = MinMaxScaler()
df[["rent_s", "bhk_s"]] = scaler.fit_transform(df[["log_rent", "BHK"]])

furn = pd.get_dummies(df["Furnishing Status"], dtype=float)

print(df[["rent_s", "bhk_s"]].describe())
print(furn.head())

            rent_s        bhk_s
count  4746.000000  4746.000000
mean      0.349402     0.216772
std       0.117407     0.166451
min       0.000000     0.000000
25%       0.265693     0.200000
50%       0.324606     0.200000
75%       0.415348     0.400000
max       1.000000     1.000000
   Furnished  Semi-Furnished  Unfurnished
0        0.0             0.0          1.0
1        0.0             1.0          0.0
2        0.0             1.0          0.0
3        0.0             0.0          1.0
4        0.0             0.0          1.0


In [ ]:
from sklearn.neighbors import NearestNeighbors

W_RENT, W_BHK, W_FURN = 3, 2, 1

def recommend(city, max_rent, bhk, furnishing, k=5):
    pool = df[(df["City"] == city) & (df["Rent"] <= max_rent)]
    if len(pool) == 0:
       return pool[["BHK", "Size", "Furnishing Status", "Rent", "Area Locality"]]
    features = pd.concat([ pool[["rent_s"]] * W_RENT, pool[["bhk_s"]] * W_BHK, furn.loc[pool.index] * W_FURN], axis=1)
    q = scaler.transform(pd.DataFrame([[np.log1p(max_rent), bhk]],columns=["log_rent", "BHK"]))[0]
    query = [q[0] * W_RENT, q[1] * W_BHK] + [W_FURN * float(c == furnishing) for c in furn.columns]
    knn = NearestNeighbors(n_neighbors=min(k, len(pool)))
    knn.fit(features.values)
    _, idx = knn.kneighbors([query])
    return pool.iloc[idx[0]][["BHK", "Size", "Furnishing Status", "Rent", "Area Locality"]]

In [9]:
print(recommend("Mumbai", 50000, 2, "Furnished"))

     BHK  Size Furnishing Status   Rent                          Area Locality
960    2  1100         Furnished  50000  Magnum Tower CHS, Lokhandwala Complex
911    2   900         Furnished  50000                              Yashodham
819    2   650         Furnished  50000                             Mahim West
805    2   950         Furnished  50000                         Ghatkopar East
775    2   935         Furnished  50000                           Andheri West


In [ ]:
recommend("Kolkata", 8000, 3, "Furnished")  -


,BHK,Size,Furnishing Status,Rent,Area Locality
288,3,1000,Furnished,7500,south station pally dankuni
249,3,800,Furnished,3200,Bhawanipur
143,2,700,Furnished,8000,Garia Dhalua Nabapally
185,2,1000,Furnished,8000,"Chinar Park, Rajarhat"
345,2,550,Furnished,7500,"New Garia Metro, New Garia"


In [12]:
print(recommend("Mumbai", 2000, 2, "Furnished"))    # budget too low
print(len(recommend("Chennai", 30000, 1, "Unfurnished")))

Empty DataFrame
Columns: [BHK, Size, Furnishing Status, Rent, Area Locality]
Index: []
5


In [ ]:
import joblib
listings = df[["City", "BHK", "Size", "Furnishing Status", "Rent", "Area Locality", "rent_s", "bhk_s"]].copy()
joblib.dump({"listings": listings, "scaler": scaler}, "recommend_model.pkl")

['recommend_model.pkl']

In [ ]:
from predict import recommend_homes
print(recommend_homes({"City": "Mumbai", "Max Rent": 50000, "BHK": 2, "Furnishing Status": "Furnished"}))
print(recommend_homes({"City": "Mumbai", "Max Rent": 2000, "BHK": 2, "Furnishing Status": "Furnished"}))

[{'BHK': 2, 'Size': 1100, 'Furnishing Status': 'Furnished', 'Rent': 50000, 'Area Locality': 'Magnum Tower CHS, Lokhandwala Complex'}, {'BHK': 2, 'Size': 900, 'Furnishing Status': 'Furnished', 'Rent': 50000, 'Area Locality': 'Yashodham'}, {'BHK': 2, 'Size': 650, 'Furnishing Status': 'Furnished', 'Rent': 50000, 'Area Locality': 'Mahim West'}, {'BHK': 2, 'Size': 950, 'Furnishing Status': 'Furnished', 'Rent': 50000, 'Area Locality': 'Ghatkopar East'}, {'BHK': 2, 'Size': 935, 'Furnishing Status': 'Furnished', 'Rent': 50000, 'Area Locality': 'Andheri West'}]
[]
